# 08. gridMET Palmer Drought Severity Index (PDSI) Operational Monthly Pipeline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/google-research/google-research/blob/master/flowracle/notebooks/data_preprocessing/08_download_gridmet_pdsi.ipynb)

This notebook downloads 4 km pentad Palmer Drought Severity Index (PDSI) grids
from the University of Idaho Northwest Knowledge Network THREDDS NetCDF Subset
Service (Abatzoglou, 2013), crops to California (`32.5–45.5°N`,
`124.0–115.0°W`), filters to pentads on or before **day 25 of each month**
(`OPERATIONAL_DAY_CUTOFF = 25`) to respect ~3–5 day NLDAS-2 latency prior to
1st-of-month forecast issuance, and writes monthly means to Zarr:
- **Output path**: `${FLOWRACLE_DATA_DIR}/pdsi-gridmet-california/pdsi_gridmet_monthly_CA.zarr`

##### Copyright 2026 The Google Research Authors.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

---
*Disclaimer: This is not an officially supported Google product.*

In [ ]:
!pip install -q "numpy>=1.26.0" "pandas>=2.0.0" "requests>=2.31.0" "beautifulsoup4>=4.12.0" "lxml>=4.9.0" "xarray>=2023.12.0" "zarr>=2.16.0,<3.0.0" "dask>=2023.12.0" "tqdm>=4.66.0" "netCDF4>=1.6.0"

In [ ]:
import datetime
import gc
import os
import tempfile

import netCDF4  # noqa: F401
import pandas as pd
import requests
from tqdm import tqdm
import xarray as xr

FLOWRACLE_DATA_DIR = os.environ.get("FLOWRACLE_DATA_DIR", "./flowracle_data")
OUTPUT_DIR = os.path.join(FLOWRACLE_DATA_DIR, "pdsi-gridmet-california")
os.makedirs(OUTPUT_DIR, exist_ok=True)
ZARR_STORE_PATH = os.path.join(OUTPUT_DIR, "pdsi_gridmet_monthly_CA.zarr")

NCSS_BASE_URL = (
    "https://thredds.northwestknowledge.net/thredds/ncss/grid/"
    "agg_met_pdsi_1979_CurrentYear_CONUS.nc"
)
LAT_MIN, LAT_MAX = 32.5, 45.5
LON_MIN, LON_MAX = -124.0, -115.0
START_YEAR = 1979
OPERATIONAL_DAY_CUTOFF = 25

In [ ]:
def download_year_via_ncss(year, temp_dir):
  params = {
      "var": "daily_mean_palmer_drought_severity_index",
      "north": LAT_MAX,
      "south": LAT_MIN,
      "west": LON_MIN,
      "east": LON_MAX,
      "time_start": f"{year}-01-01T00:00:00Z",
      "time_end": f"{year}-12-31T00:00:00Z",
      "accept": "netcdf",
  }
  local_path = os.path.join(temp_dir, f"pdsi_{year}.nc")
  resp = requests.get(NCSS_BASE_URL, params=params, stream=True, timeout=300)
  resp.raise_for_status()
  with open(local_path, "wb") as f:
    for chunk in resp.iter_content(chunk_size=8192):
      f.write(chunk)
  with xr.open_dataset(local_path) as raw_ds:
    ds = raw_ds.load()
  os.remove(local_path)
  if "day" in ds.dims and "time" not in ds.dims:
    ds = ds.rename({"day": "time"})
  return ds


def resample_to_operational_monthly(ds):
  ds_causal = ds.sel(time=(ds.time.dt.day <= OPERATIONAL_DAY_CUTOFF))
  return ds_causal.resample(time="MS").mean(skipna=True)


def append_pdsi_zarr(ds_monthly, zarr_path):
  if not os.path.exists(zarr_path):
    enc = {
        v: {"chunks": (12,) + ds_monthly[v].shape[1:]}
        for v in ds_monthly.data_vars
    }
    ds_monthly.to_zarr(zarr_path, mode="w", encoding=enc)
  else:
    ds_monthly.to_zarr(zarr_path, mode="a", append_dim="time")

In [ ]:
current_year = datetime.datetime.now().year
existing_years = set()
if os.path.exists(ZARR_STORE_PATH):
  ds_ex = xr.open_zarr(ZARR_STORE_PATH)
  existing_years = set(pd.to_datetime(ds_ex.time.values).year)
  ds_ex.close()

years_todo = [
    y for y in range(START_YEAR, current_year + 1) if y not in existing_years
]
with tempfile.TemporaryDirectory() as temp_dir:
  for year in tqdm(years_todo, desc="Downloading gridMET PDSI"):
    ds_yr = download_year_via_ncss(year, temp_dir)
    ds_mo = resample_to_operational_monthly(ds_yr)
    var0 = list(ds_mo.data_vars)[0]
    valid_mask = ds_mo[var0].notnull().any(dim=["lat", "lon"])
    ds_mo = ds_mo.sel(time=valid_mask)
    if ds_mo.sizes["time"] > 0:
      append_pdsi_zarr(ds_mo, ZARR_STORE_PATH)
    del ds_yr, ds_mo
    gc.collect()

print(xr.open_zarr(ZARR_STORE_PATH))